# Test the SHAR classifier on your own video

Runs the evaluated D-53 classifier (frozen DINOv2-B, attention head) on a video you choose and shows the report.

**Start it from the head node** with `scripts/dgx/notebook.sh`, which opens this inside a pod on the GPU slice and prints the
connection steps. The notebook holds the slice while it is open, so stop the pod when you finish. For a single quick test,
`scripts/dgx/test_video.sh my_videos/clip.mp4` does the same job and frees the slice by itself.

**Put your video** under the home directory, for example in `my_videos/` at the repository root.

**Reading the answer:** the model always picks one of its 14 classes, and it has no "none of these". It scored macro-F1 0.414 and
accuracy 41.8% on held-out UCF-Crime test videos, against inherited video-level labels (not event-only). A few chosen videos are
examples, not a measurement. Phone or movie footage is outside what was evaluated.

## 1. Choose the video

Set `VIDEO` to a file or a folder. Set `LABEL` to the true class if you know it, otherwise leave it as `None`.

In [ ]:
import os

VIDEO = os.environ.get("SHAR_TEST_VIDEO", "my_videos/")  # a file or a folder, relative to the repository root
LABEL = None  # one video only, for example "Fighting"; see the class list below
LABELS_CSV = os.environ.get("SHAR_TEST_LABELS") or None  # for a folder: a CSV with the columns video,label

CLASSES = ["Abuse", "Arrest", "Arson", "Assault", "Burglary", "Explosion", "Fighting", "Normal", "RoadAccidents",
           "Robbery", "Shooting", "Shoplifting", "Stealing", "Vandalism"]

## 2. Run it

The model takes a few seconds to load each time this cell runs, then each video takes a few seconds on the GPU.

In [ ]:
import datetime as dt
import html
import json
import subprocess
from pathlib import Path

from IPython.display import HTML, display

root = Path.cwd()
while not (root / "scripts/test_video.py").exists():
    if root == root.parent:
        raise FileNotFoundError("start this notebook from inside the SHAR repository")
    root = root.parent
python = os.environ.get("SHAR_PYTHON", str(root / ".venv/bin/python"))
out_dir = root / "results/video-tests" / ("notebook-" + dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ"))

command = [python, "scripts/test_video.py", VIDEO, "--out", str(out_dir)]
if LABEL:
    command += ["--label", LABEL]
if LABELS_CSV:
    command += ["--labels", LABELS_CSV]
run = subprocess.run(command, cwd=root, capture_output=True, text=True)
print(run.stdout)
if run.returncode:
    raise RuntimeError(run.stderr[-3000:] or "the test failed; see the output above")

## 3. See the report

The same page is saved as `report.html` in the output folder, next to `results.json` and `results.csv`.

In [ ]:
report = (out_dir / "report.html").read_text(encoding="utf-8")
display(HTML(f'<iframe srcdoc="{html.escape(report)}" style="width:100%;height:900px;border:0"></iframe>'))
print("Saved in", out_dir)

## 4. Results as data

In [ ]:
data = json.loads((out_dir / "results.json").read_text(encoding="utf-8"))
print(json.dumps(data["summary"], indent=1))
for row in data["results"]:
    if row.get("error"):
        print(f"{row['filename']:40s} ERROR {row['error']}")
    else:
        print(f"{row['filename']:40s} {row['prediction']:20s} {row['confidence']:6.1%}  label={row.get('label')}  correct={row.get('correct')}")